In [ ]:
import pandas as pd
import numpy as np
import cv2
import xml
import xml.etree.ElementTree as ET
import os
import matplotlib.pyplot as plt
from skimage.transform import resize
import math
import warnings
from scipy import ndimage

In [ ]:
# Functions for creating the input features subwindow stacks

In [ ]:
def chose_xml_and_jpeg(file_location):
    # list all files in location
    list_of_all_files = os.listdir(file_location)
    # sort files
    list_of_all_files.sort()
    # separate xml and jpeg files
    all_xml_files = [file for file in list_of_all_files if file.split('.')[-1] == 'xml']
    all_jpeg_files = [file for file in list_of_all_files if file not in all_xml_files]
    # get the final 20 files
    chosen_xml_files = all_xml_files[-20:]
    chosen_jpeg_files = all_jpeg_files[-20:]
    # make sure the xml and jpeg files correspond to each other?
    mean = np.mean([file.split('.')[0] for file in chosen_xml_files] == [file.split('.')[0] for file in chosen_jpeg_files])

    # chose the required files only
    task_specific_image_files = chosen_jpeg_files[:13]
    task_specific_xml_files = chosen_xml_files[-7:]
    return(task_specific_image_files, task_specific_xml_files, mean)

In [ ]:
def create_and_stack_subwindows(im_height, im_width, stride, kernel_size, n_channels, im_list):
    # initiate an empty list
    catch_all_subwindows = []
    sub_window_name_list = []
    counter = 0
    for i in range(0, im_height, stride):
        for j in range(0, im_width, stride):
            local_list_at_subimage_sequence_level= []
            for im_file in im_list:
                chosen_window = im_file[i:i+kernel_size, j:j+kernel_size, :]
                # resize the window
                chosen_window = resize(chosen_window, (kernel_size, kernel_size ,n_channels))
                local_list_at_subimage_sequence_level.append(chosen_window)
            catch_all_subwindows.append(local_list_at_subimage_sequence_level)
            sub_window_name_list.append("sub_image_seq_" + str(counter))
            counter = counter + 1
    # stack all these together
    Stacked_subwindows = np.stack(catch_all_subwindows)
    return(Stacked_subwindows, sub_window_name_list)

In [ ]:
# Functions for creating the required response stacks

In [ ]:
# need the function for storing the np files
def store_images_as_np_arrays_horizontal(img_old_path, img_name, img_store_path):
    # join the path
    image_path = os.path.join(img_old_path, img_name)
    # read the image
    read_image = plt.imread(image_path)
    image_size = read_image.shape
    # show the image
    plt.imshow(read_image)
    plt.show()
    # save the image in new location
    np.save(img_store_path + '/' + img_name.split(".")[0] + '.npy', read_image)
    return(image_size)

In [ ]:
# create density maps for the horizontally annotated images
def get_density_maps_horizontal(file_name, image_path, xml_path, save_density_path):
    # we need a list just with the names of the files and not the .xml or .jpeg extensions
    xml_file = file_name + '.xml'
    xml_file_path = os.path.join(xml_path, xml_file)

    # Get coords from the xml file
    # parse the xml file
    parsed_file = ET.parse(xml_file_path)
    # get the roots
    root = parsed_file.getroot()
    # get the roots here
    coords = []
    for child in root:
        for i in child:
            for j in i:
                coords.append(int(j.text))
    
    # chunk the points into sets of 4 - these are the coordinates of the bounding boxes
    points_tupples = []
    for i in range(0, len(coords), 4):
        points_tupples.append(coords[i:i + 4])

    # make a dataframe with these points
    coords_df = pd.DataFrame(points_tupples, columns = ["bleft_x", "bleft_y", "tright_x", "tright_y"])

    # compute the number of tassels in each image
    no_of_tassels = len(points_tupples)

    # compute the mid coordinates
    coords_df["mid_x"] = (round(0.5*(coords_df["bleft_x"] + coords_df["tright_x"]))).astype(int)
    coords_df["mid_y"] = (round(0.5*(coords_df["bleft_y"] + coords_df["tright_y"]))).astype(int)

    # extract the mid cordinates
    mid_coords = coords_df[["mid_x", "mid_y"]]
    # cap the coords at the max height and width values
    mid_coords.loc[mid_coords['mid_x'] > 1024, 'mid_x'] = 1023
    mid_coords.loc[mid_coords['mid_y'] > 768, 'mid_y'] = 767
    warnings.filterwarnings("ignore")

    # plot the bounding boxes on images
    # get image name and path
    image_name = file_name + '.npy'
    imge_file_path = os.path.join(image_path, image_name)
    # read the image
    read_image = np.load(imge_file_path)
    # check the shape of the read image
    read_image_shape = read_image.shape
    #  plot the bounding boxes on the image
    for points in points_tupples:
        annotated_image = cv2.rectangle(read_image, (points[0],points[1]), (points[2],points[3]), color = (255,0,0), thickness = 2)
    # plt.figure(figsize = (12,18))
    plt.imshow(annotated_image)
    plt.show()

    # plot the mid points on the image
    coords_list = mid_coords.values.tolist()
    # read the image again
    read_image_again = np.load(imge_file_path)
    # draw the circles on image
    for i in coords_list:
        image_with_mids = cv2.circle(read_image_again, i, radius=5, color=(255, 0, 0), thickness=-1)
    # look at the annotated image
    # plt.figure(figsize = (12,18))
    plt.imshow(image_with_mids)
    plt.show()

    # also try creating the density map here
    # first create the empty maps
    np_image = np.zeros((read_image_shape[0], read_image_shape[1]))
    # get the dot maps
    for point in coords_list:
        np_image[point[1], point[0]] = 1
    # plot the image
    # plt.figure(figsize = (12,18))
    plt.imshow(np_image, cmap = "Greys")
    plt.show()

    # now define the kernel and run the convolution
    one_d_kerenel = cv2.getGaussianKernel(50,5)
    two_d_kernel = np.multiply(one_d_kerenel.T, one_d_kerenel)

    # Shape of the 2D kernel
    twoD_shape = two_d_kernel.shape
        
    # do the convolution
    convolution = ndimage.convolve(np_image, two_d_kernel)
        
    # plot the density map
    # plt.figure(figsize = (12,18))
    plt.imshow(convolution, cmap = "Greys")
    plt.show()
        
    # get the sums of the images
    img_sum = np.sum(convolution)

    # save the density map
    np.save(save_density_path + '/' + file_name + '_density_map.npy', convolution)

    return(file_name, read_image_shape, no_of_tassels, img_sum, convolution)

In [ ]:
# And the one for creating the sub-window tassel density sequences

def tassel_density_sequences(im_height, im_width, stride, kernel_size, file_seq):

    # empty lsit for catching the densities
    catch_tassel_densities_all = []
    sub_seq_name_list = []
    counter = 0
    for i in range(0, im_height, stride):
        for j in range(0, im_width, stride):
            Get_this_to_work = []
            for file in file_seq:
                chosen_density_window = file[i:i+kernel_size, j:j+kernel_size]
                # resize the window
                sub_window_desity = np.sum(chosen_density_window)
                Get_this_to_work.append(sub_window_desity)
            catch_tassel_densities_all.append(Get_this_to_work)
            sub_seq_name_list.append('sub_dense_seq_' + str(counter))
            counter = counter + 1
    stacked_responses = np.stack(catch_tassel_densities_all)
    return(stacked_responses, sub_seq_name_list)

Block 0101

In [ ]:
# location of the files
block_0101 = '../../../Spring_2024/S_lab_TasselNet/Block_1_TN/Block_1_images_and_xml'

In [ ]:
task_spec_im_files_0101, task_spec_xml_files_0101, mean_0101 = chose_xml_and_jpeg(block_0101)

In [ ]:
# task_spec_im_files_0101

In [ ]:
# task_spec_xml_files_0101

In [ ]:
# mean_0101

In [ ]:
images_0101 = []
for file in task_spec_im_files_0101:
    joined_path = os.path.join(block_0101, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0101.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0101[0].shape[0]
image_weight = images_0101[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0101, sub_window_name_list = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0101)

In [ ]:
len(sub_window_name_list)

In [ ]:
sub_window_name_list[0], sub_window_name_list[1], sub_window_name_list[-1]

In [ ]:
stacked_sub_windows_0101.shape

In [ ]:
np.save('All_data_alt/block_0101/subwindow_seqs_0101.npy', stacked_sub_windows_0101)

In [ ]:
# xml files
task_spec_xml_files_0101

In [ ]:
# Corresponding image files
response_image_files_0101 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0101]

In [ ]:
response_image_files_0101

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0101.sort()
response_image_files_0101.sort()

In [ ]:
len(task_spec_xml_files_0101), len(response_image_files_0101)

In [ ]:
old_path = block_0101
old_path

In [ ]:
new_store_path_0101 = 'All_data_alt/block_0101/all_np_files'

In [ ]:
%%time
all_im_sizes_0101 = []
for file in response_image_files_0101:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0101)
    all_im_sizes_0101.append(im_size)

In [ ]:
all_im_sizes_0101

In [ ]:
# we need file names
block_0101_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0101]
block_0101_response_file_names

In [ ]:
new_store_path_0101

In [ ]:
block_0101

In [ ]:
%%time
# let's run the function
function_results_0101 = []
for file_name in block_0101_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0101, block_0101, new_store_path_0101)
    function_results_0101.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0101 = [file[2] for file in function_results_0101]
convolved_counts_block_0101 = [file[3] for file in function_results_0101]

In [ ]:
true_counts_block_0101

In [ ]:
convolved_counts_block_0101

In [ ]:
np.mean(true_counts_block_0101 == np.round(convolved_counts_block_0101))

In [ ]:
all_density_maps_0101 = [file[-1] for file in function_results_0101]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0101 == [np.sum(file) for file in all_density_maps_0101])

In [ ]:
# save the true counts here
image_names_blk_0101 = [file[0] for file in function_results_0101]

In [ ]:
true_counts_df_block_0101 = pd.DataFrame({'Image_name' :image_names_blk_0101, 'True_count': true_counts_block_0101, 'Convolved_count': convolved_counts_block_0101})

In [ ]:
# save this dataframe
true_counts_df_block_0101.to_csv("All_data_alt/test_true_counts/true_counts_blk_0101.csv", index = False)

In [ ]:
im_height = all_im_sizes_0101[0][0]
im_width = all_im_sizes_0101[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0101, response_name_list_0101 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0101)

In [ ]:
stacked_responses_0101.shape

In [ ]:
response_name_list_0101[0], response_name_list_0101[1], response_name_list_0101[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0101/density_maps_seqs_0101.npy', stacked_responses_0101)

Block 0102

In [ ]:
# location of the files
block_0102 = '../../../Spring_2024/S_lab_TasselNet/Block_2_TN/Block_2_images_and_xml'

In [ ]:
task_spec_im_files_0102, task_spec_xml_files_0102, mean_0102 = chose_xml_and_jpeg(block_0102)

In [ ]:
# task_spec_im_files_0102

In [ ]:
# task_spec_xml_files_0102

In [ ]:
# mean_0102

In [ ]:
images_0102 = []
for file in task_spec_im_files_0102:
    joined_path = os.path.join(block_0102, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0102.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0102[0].shape[0]
image_weight = images_0102[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0102, sub_window_name_list = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0102)

In [ ]:
len(sub_window_name_list)

In [ ]:
sub_window_name_list[0], sub_window_name_list[1], sub_window_name_list[-1]

In [ ]:
stacked_sub_windows_0102.shape

In [ ]:
np.save('All_data_alt/block_0102/subwindow_seqs_0102.npy', stacked_sub_windows_0102)

In [ ]:
# xml files
task_spec_xml_files_0102

In [ ]:
# Corresponding image files
response_image_files_0102 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0102]

In [ ]:
response_image_files_0102

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0102.sort()
response_image_files_0102.sort()

In [ ]:
len(task_spec_xml_files_0102), len(response_image_files_0102)

In [ ]:
old_path = block_0102
old_path

In [ ]:
new_store_path_0102 = 'All_data_alt/block_0102/all_np_files'

In [ ]:
%%time
all_im_sizes_0102 = []
for file in response_image_files_0102:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0102)
    all_im_sizes_0102.append(im_size)

In [ ]:
all_im_sizes_0102

In [ ]:
# we need file names
block_0102_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0102]
block_0102_response_file_names

In [ ]:
new_store_path_0102

In [ ]:
block_0102

In [ ]:
%%time
# let's run the function
function_results_0102 = []
for file_name in block_0102_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0102, block_0102, new_store_path_0102)
    function_results_0102.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0102 = [file[2] for file in function_results_0102]
convolved_counts_block_0102 = [file[3] for file in function_results_0102]

In [ ]:
true_counts_block_0102

In [ ]:
convolved_counts_block_0102

In [ ]:
np.mean(true_counts_block_0102 == np.round(convolved_counts_block_0102))

In [ ]:
all_density_maps_0102 = [file[-1] for file in function_results_0102]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0102 == [np.sum(file) for file in all_density_maps_0102])

In [ ]:
# save the true counts here
image_names_blk_0102 = [file[0] for file in function_results_0102]

In [ ]:
true_counts_df_block_0102 = pd.DataFrame({'Image_name' :image_names_blk_0102, 'True_count': true_counts_block_0102, 'Convolved_count': convolved_counts_block_0102})

In [ ]:
true_counts_df_block_0102

In [ ]:
# save this dataframe
true_counts_df_block_0102.to_csv("All_data_alt/test_true_counts/true_counts_blk_0102.csv", index = False)

In [ ]:
im_height = all_im_sizes_0102[0][0]
im_width = all_im_sizes_0102[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0102, response_name_list_0102 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0102)

In [ ]:
stacked_responses_0102.shape

In [ ]:
response_name_list_0102[0], response_name_list_0102[1], response_name_list_0102[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0102/density_maps_seqs_0102.npy', stacked_responses_0102)

In [ ]:
# Let's continue this for the rest of the blocks

In [ ]:
# For just the testing blocks, do we need these preprocessed in some other way? We need the inputs as they are, and we do not necessarily need the density maps, but let's keep these anyway, just incase they come useful - like maybe to compute the true counts.

Block 0103

In [ ]:
# location of the files
block_0103 = '../../../Spring_2024/S_lab_TasselNet/Block_3_TN/Block_3_images_and_xml'

In [ ]:
task_spec_im_files_0103, task_spec_xml_files_0103, mean_0103 = chose_xml_and_jpeg(block_0103)

In [ ]:
# task_spec_im_files_0103

In [ ]:
# task_spec_xml_files_0103

In [ ]:
# mean_0103

In [ ]:
images_0103 = []
for file in task_spec_im_files_0103:
    joined_path = os.path.join(block_0103, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0103.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0103[0].shape[0]
image_weight = images_0103[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0103, sub_window_name_list_0103 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0103)

In [ ]:
len(sub_window_name_list_0103)

In [ ]:
sub_window_name_list_0103[0], sub_window_name_list_0103[1], sub_window_name_list_0103[-1]

In [ ]:
stacked_sub_windows_0103.shape

In [ ]:
np.save('All_data_alt/block_0103/subwindow_seqs_0103.npy', stacked_sub_windows_0103)

In [ ]:
# xml files
task_spec_xml_files_0103

In [ ]:
# Corresponding image files
response_image_files_0103 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0103]

In [ ]:
response_image_files_0103

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0103.sort()
response_image_files_0103.sort()

In [ ]:
len(task_spec_xml_files_0103), len(response_image_files_0103)

In [ ]:
old_path = block_0103
old_path

In [ ]:
new_store_path_0103 = 'All_data_alt/block_0103/all_np_files'

In [ ]:
%%time
all_im_sizes_0103 = []
for file in response_image_files_0103:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0103)
    all_im_sizes_0103.append(im_size)

In [ ]:
all_im_sizes_0103

In [ ]:
# we need file names
block_0103_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0103]
block_0103_response_file_names

In [ ]:
new_store_path_0103

In [ ]:
block_0103

In [ ]:
%%time
# let's run the function
function_results_0103 = []
for file_name in block_0103_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0103, block_0103, new_store_path_0103)
    function_results_0103.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0103 = [file[2] for file in function_results_0103]
convolved_counts_block_0103 = [file[3] for file in function_results_0103]

In [ ]:
true_counts_block_0103

In [ ]:
convolved_counts_block_0103

In [ ]:
np.mean(true_counts_block_0103 == np.round(convolved_counts_block_0103))

In [ ]:
all_density_maps_0103 = [file[-1] for file in function_results_0103]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0103 == [np.sum(file) for file in all_density_maps_0103])

In [ ]:
# save the true counts here
image_names_blk_0103 = [file[0] for file in function_results_0103]

In [ ]:
true_counts_df_block_0103 = pd.DataFrame({'Image_name' :image_names_blk_0103, 'True_count': true_counts_block_0103, 'Convolved_count': convolved_counts_block_0103})

In [ ]:
true_counts_df_block_0103

In [ ]:
# save this dataframe
true_counts_df_block_0103.to_csv("All_data_alt/test_true_counts/true_counts_blk_0103.csv", index = False)

In [ ]:
im_height = all_im_sizes_0103[0][0]
im_width = all_im_sizes_0103[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0103, response_name_list_0103 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0103)

In [ ]:
stacked_responses_0103.shape

In [ ]:
response_name_list_0103[0], response_name_list_0103[1], response_name_list_0103[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0103/density_maps_seqs_0103.npy', stacked_responses_0103)

Block 0104

In [ ]:
# location of the files
block_0104 = '../../../Spring_2024/S_lab_TasselNet/Block_4_TN/Block_4_images_and_xml'

In [ ]:
task_spec_im_files_0104, task_spec_xml_files_0104, mean_0104 = chose_xml_and_jpeg(block_0104)

In [ ]:
# task_spec_im_files_0104

In [ ]:
# task_spec_xml_files_0104

In [ ]:
# mean_0104

In [ ]:
images_0104 = []
for file in task_spec_im_files_0104:
    joined_path = os.path.join(block_0104, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0104.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0104[0].shape[0]
image_weight = images_0104[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0104, sub_window_name_list_0104 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0104)

In [ ]:
len(sub_window_name_list_0104)

In [ ]:
sub_window_name_list_0104[0], sub_window_name_list_0104[1], sub_window_name_list_0104[-1]

In [ ]:
stacked_sub_windows_0104.shape

In [ ]:
np.save('All_data_alt/block_0104/subwindow_seqs_0104.npy', stacked_sub_windows_0104)

In [ ]:
# xml files
task_spec_xml_files_0104

In [ ]:
# Corresponding image files
response_image_files_0104 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0104]

In [ ]:
response_image_files_0104

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0104.sort()
response_image_files_0104.sort()

In [ ]:
len(task_spec_xml_files_0104), len(response_image_files_0104)

In [ ]:
old_path = block_0104
old_path

In [ ]:
new_store_path_0104 = 'All_data_alt/block_0104/all_np_files'

In [ ]:
%%time
all_im_sizes_0104 = []
for file in response_image_files_0104:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0104)
    all_im_sizes_0104.append(im_size)

In [ ]:
all_im_sizes_0104

In [ ]:
# we need file names
block_0104_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0104]
block_0104_response_file_names

In [ ]:
new_store_path_0104

In [ ]:
block_0104

In [ ]:
%%time
# let's run the function
function_results_0104 = []
for file_name in block_0104_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0104, block_0104, new_store_path_0104)
    function_results_0104.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0104 = [file[2] for file in function_results_0104]
convolved_counts_block_0104 = [file[3] for file in function_results_0104]

In [ ]:
true_counts_block_0104

In [ ]:
convolved_counts_block_0104

In [ ]:
np.mean(true_counts_block_0104 == np.round(convolved_counts_block_0104))

In [ ]:
all_density_maps_0104 = [file[-1] for file in function_results_0104]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0104 == [np.sum(file) for file in all_density_maps_0104])

In [ ]:
# save the true counts here
image_names_blk_0104 = [file[0] for file in function_results_0104]

In [ ]:
true_counts_df_block_0104 = pd.DataFrame({'Image_name' :image_names_blk_0104, 'True_count': true_counts_block_0104, 'Convolved_count': convolved_counts_block_0104})

In [ ]:
true_counts_df_block_0104

In [ ]:
# save this dataframe
true_counts_df_block_0104.to_csv("All_data_alt/test_true_counts/true_counts_blk_0104.csv", index = False)

In [ ]:
im_height = all_im_sizes_0104[0][0]
im_width = all_im_sizes_0104[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0104, response_name_list_0104 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0104)

In [ ]:
stacked_responses_0104.shape

In [ ]:
response_name_list_0104[0], response_name_list_0104[1], response_name_list_0104[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0104/density_maps_seqs_0104.npy', stacked_responses_0104)

Block 0105

In [ ]:
# location of the files
block_0105 = '../../../Spring_2024/S_lab_TasselNet/Block_5_TN/Block_5_images_and_xml'

In [ ]:
task_spec_im_files_0105, task_spec_xml_files_0105, mean_0105 = chose_xml_and_jpeg(block_0105)

In [ ]:
# task_spec_im_files_0105

In [ ]:
# task_spec_xml_files_0105

In [ ]:
# mean_0105

In [ ]:
images_0105 = []
for file in task_spec_im_files_0105:
    joined_path = os.path.join(block_0105, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0105.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0105[0].shape[0]
image_weight = images_0105[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0105, sub_window_name_list_0105 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0105)

In [ ]:
len(sub_window_name_list_0105)

In [ ]:
sub_window_name_list_0105[0], sub_window_name_list_0105[1], sub_window_name_list_0105[-1]

In [ ]:
stacked_sub_windows_0105.shape

In [ ]:
np.save('All_data_alt/block_0105/subwindow_seqs_0105.npy', stacked_sub_windows_0105)

In [ ]:
# xml files
task_spec_xml_files_0105

In [ ]:
# Corresponding image files
response_image_files_0105 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0105]

In [ ]:
response_image_files_0105

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0105.sort()
response_image_files_0105.sort()

In [ ]:
len(task_spec_xml_files_0105), len(response_image_files_0105)

In [ ]:
old_path = block_0105
old_path

In [ ]:
new_store_path_0105 = 'All_data_alt/block_0105/all_np_files'

In [ ]:
%%time
all_im_sizes_0105 = []
for file in response_image_files_0105:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0105)
    all_im_sizes_0105.append(im_size)

In [ ]:
all_im_sizes_0105

In [ ]:
# we need file names
block_0105_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0105]
block_0105_response_file_names

In [ ]:
new_store_path_0105

In [ ]:
block_0105

In [ ]:
%%time
# let's run the function
function_results_0105 = []
for file_name in block_0105_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0105, block_0105, new_store_path_0105)
    function_results_0105.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0105 = [file[2] for file in function_results_0105]
convolved_counts_block_0105 = [file[3] for file in function_results_0105]

In [ ]:
true_counts_block_0105

In [ ]:
convolved_counts_block_0105

In [ ]:
np.mean(true_counts_block_0105 == np.round(convolved_counts_block_0105))

In [ ]:
all_density_maps_0105 = [file[-1] for file in function_results_0105]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0105 == [np.sum(file) for file in all_density_maps_0105])

In [ ]:
# save the true counts here
image_names_blk_0105 = [file[0] for file in function_results_0105]

In [ ]:
true_counts_df_block_0105 = pd.DataFrame({'Image_name' :image_names_blk_0105, 'True_count': true_counts_block_0105, 'Convolved_count': convolved_counts_block_0105})

In [ ]:
true_counts_df_block_0105

In [ ]:
# save this dataframe
true_counts_df_block_0105.to_csv("All_data_alt/test_true_counts/true_counts_blk_0105.csv", index = False)

In [ ]:
im_height = all_im_sizes_0105[0][0]
im_width = all_im_sizes_0105[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0105, response_name_list_0105 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0105)

In [ ]:
stacked_responses_0105.shape

In [ ]:
response_name_list_0105[0], response_name_list_0105[1], response_name_list_0105[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0105/density_maps_seqs_0105.npy', stacked_responses_0105)

Block 0106

In [ ]:
# location of the files
block_0106 = '../../../Spring_2024/S_lab_TasselNet/Block_6_TN/Block_6_images_and_xml'

In [ ]:
task_spec_im_files_0106, task_spec_xml_files_0106, mean_0106 = chose_xml_and_jpeg(block_0106)

In [ ]:
# task_spec_im_files_0106

In [ ]:
# task_spec_xml_files_0106

In [ ]:
# mean_0106

In [ ]:
images_0106 = []
for file in task_spec_im_files_0106:
    joined_path = os.path.join(block_0106, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0106.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0106[0].shape[0]
image_weight = images_0106[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0106, sub_window_name_list_0106 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0106)

In [ ]:
len(sub_window_name_list_0106)

In [ ]:
sub_window_name_list_0106[0], sub_window_name_list_0106[1], sub_window_name_list_0106[-1]

In [ ]:
stacked_sub_windows_0106.shape

In [ ]:
np.save('All_data_alt/block_0106/subwindow_seqs_0106.npy', stacked_sub_windows_0106)

In [ ]:
# xml files
task_spec_xml_files_0106

In [ ]:
# Corresponding image files
response_image_files_0106 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0106]

In [ ]:
response_image_files_0106

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0106.sort()
response_image_files_0106.sort()

In [ ]:
len(task_spec_xml_files_0106), len(response_image_files_0106)

In [ ]:
old_path = block_0106
old_path

In [ ]:
new_store_path_0106 = 'All_data_alt/block_0106/all_np_files'

In [ ]:
%%time
all_im_sizes_0106 = []
for file in response_image_files_0106:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0106)
    all_im_sizes_0106.append(im_size)

In [ ]:
all_im_sizes_0106

In [ ]:
# we need file names
block_0106_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0106]
block_0106_response_file_names

In [ ]:
new_store_path_0106

In [ ]:
block_0106

In [ ]:
%%time
# let's run the function
function_results_0106 = []
for file_name in block_0106_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0106, block_0106, new_store_path_0106)
    function_results_0106.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0106 = [file[2] for file in function_results_0106]
convolved_counts_block_0106 = [file[3] for file in function_results_0106]

In [ ]:
true_counts_block_0106

In [ ]:
convolved_counts_block_0106

In [ ]:
np.mean(true_counts_block_0106 == np.round(convolved_counts_block_0106))

In [ ]:
all_density_maps_0106 = [file[-1] for file in function_results_0106]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0106 == [np.sum(file) for file in all_density_maps_0106])

In [ ]:
# save the true counts here
image_names_blk_0106 = [file[0] for file in function_results_0106]

In [ ]:
true_counts_df_block_0106 = pd.DataFrame({'Image_name' :image_names_blk_0106, 'True_count': true_counts_block_0106, 'Convolved_count': convolved_counts_block_0106})

In [ ]:
true_counts_df_block_0106

In [ ]:
# save this dataframe
true_counts_df_block_0106.to_csv("All_data_alt/test_true_counts/true_counts_blk_0106.csv", index = False)

In [ ]:
im_height = all_im_sizes_0106[0][0]
im_width = all_im_sizes_0106[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0106, response_name_list_0106 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0106)

In [ ]:
stacked_responses_0106.shape

In [ ]:
response_name_list_0106[0], response_name_list_0106[1], response_name_list_0106[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0106/density_maps_seqs_0106.npy', stacked_responses_0106)

In [ ]:
# Note that we had done everything to all the blocks, but we lost some of the work with some git issues. It's okay, as we had preprocessed all data. But I'm not sure if we have the correct data - Should we do this again? It's gonna take some time But it's okay we will also store the true counts here itself. Let's do this from the begining. Save the new work in a separate folder to ensure what we indeed had previously is correct.

Block 0201

In [ ]:
# location of the files
block_0201 = '../../../Spring_2024/S_lab_TasselNet/Block_7_TN/Block_7_images_and_xml'

In [ ]:
task_spec_im_files_0201, task_spec_xml_files_0201, mean_0201 = chose_xml_and_jpeg(block_0201)

In [ ]:
# task_spec_im_files_0201

In [ ]:
# task_spec_xml_files_0201

In [ ]:
# mean_0201

In [ ]:
images_0201 = []
for file in task_spec_im_files_0201:
    joined_path = os.path.join(block_0201, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0201.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0201[0].shape[0]
image_weight = images_0201[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0201, sub_window_name_list = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0201)

In [ ]:
len(sub_window_name_list)

In [ ]:
sub_window_name_list[0], sub_window_name_list[1], sub_window_name_list[-1]

In [ ]:
stacked_sub_windows_0201.shape

In [ ]:
np.save('All_data_alt/block_0201/subwindow_seqs_0201.npy', stacked_sub_windows_0201)

In [ ]:
# xml files
task_spec_xml_files_0201

In [ ]:
# Corresponding image files
response_image_files_0201 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0201]

In [ ]:
response_image_files_0201

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0201.sort()
response_image_files_0201.sort()

In [ ]:
len(task_spec_xml_files_0201), len(response_image_files_0201)

In [ ]:
old_path = block_0201
old_path

In [ ]:
new_store_path_0201 = 'All_data_alt/block_0201/all_np_files'

In [ ]:
%%time
all_im_sizes_0201 = []
for file in response_image_files_0201:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0201)
    all_im_sizes_0201.append(im_size)

In [ ]:
all_im_sizes_0201

In [ ]:
# we need file names
block_0201_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0201]
block_0201_response_file_names

In [ ]:
new_store_path_0201

In [ ]:
block_0201

In [ ]:
%%time
# let's run the function
function_results_0201 = []
for file_name in block_0201_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0201, block_0201, new_store_path_0201)
    function_results_0201.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0201 = [file[2] for file in function_results_0201]
convolved_counts_block_0201 = [file[3] for file in function_results_0201]

In [ ]:
true_counts_block_0201

In [ ]:
convolved_counts_block_0201

In [ ]:
np.mean(true_counts_block_0201 == np.round(convolved_counts_block_0201))

In [ ]:
all_density_maps_0201 = [file[-1] for file in function_results_0201]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0201 == [np.sum(file) for file in all_density_maps_0201])

In [ ]:
# save the true counts here
image_names_blk_0201 = [file[0] for file in function_results_0201]

In [ ]:
true_counts_df_block_0201 = pd.DataFrame({'Image_name' :image_names_blk_0201, 'True_count': true_counts_block_0201, 'Convolved_count': convolved_counts_block_0201})

In [ ]:
true_counts_df_block_0201

In [ ]:
# save this dataframe
true_counts_df_block_0201.to_csv("All_data_alt/test_true_counts/true_counts_blk_0201.csv", index = False)

In [ ]:
im_height = all_im_sizes_0201[0][0]
im_width = all_im_sizes_0201[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0201, response_name_list_0201 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0201)

In [ ]:
stacked_responses_0201.shape

In [ ]:
response_name_list_0201[0], response_name_list_0201[1], response_name_list_0201[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0201/density_maps_seqs_0201.npy', stacked_responses_0201)

Block 0202

In [ ]:
# location of the files
block_0202 = '../../../Spring_2024/S_lab_TasselNet/Block_8_TN/Block_8_images_and_xml'

In [ ]:
task_spec_im_files_0202, task_spec_xml_files_0202, mean_0202 = chose_xml_and_jpeg(block_0202)

In [ ]:
# task_spec_im_files_0202

In [ ]:
# task_spec_xml_files_0202

In [ ]:
# mean_0202

In [ ]:
images_0202 = []
for file in task_spec_im_files_0202:
    joined_path = os.path.join(block_0202, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0202.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0202[0].shape[0]
image_weight = images_0202[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0202, sub_window_name_list = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0202)

In [ ]:
len(sub_window_name_list)

In [ ]:
sub_window_name_list[0], sub_window_name_list[1], sub_window_name_list[-1]

In [ ]:
stacked_sub_windows_0202.shape

In [ ]:
np.save('All_data_alt/block_0202/subwindow_seqs_0202.npy', stacked_sub_windows_0202)

In [ ]:
# xml files
task_spec_xml_files_0202

In [ ]:
# Corresponding image files
response_image_files_0202 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0202]

In [ ]:
response_image_files_0202

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0202.sort()
response_image_files_0202.sort()

In [ ]:
len(task_spec_xml_files_0202), len(response_image_files_0202)

In [ ]:
old_path = block_0202
old_path

In [ ]:
new_store_path_0202 = 'All_data_alt/block_0202/all_np_files'

In [ ]:
%%time
all_im_sizes_0202 = []
for file in response_image_files_0202:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0202)
    all_im_sizes_0202.append(im_size)

In [ ]:
all_im_sizes_0202

In [ ]:
# we need file names
block_0202_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0202]
block_0202_response_file_names

In [ ]:
new_store_path_0202

In [ ]:
block_0202

In [ ]:
%%time
# let's run the function
function_results_0202 = []
for file_name in block_0202_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0202, block_0202, new_store_path_0202)
    function_results_0202.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0202 = [file[2] for file in function_results_0202]
convolved_counts_block_0202 = [file[3] for file in function_results_0202]

In [ ]:
true_counts_block_0202

In [ ]:
convolved_counts_block_0202

In [ ]:
np.mean(true_counts_block_0202 == np.round(convolved_counts_block_0202))

In [ ]:
all_density_maps_0202 = [file[-1] for file in function_results_0202]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0202 == [np.sum(file) for file in all_density_maps_0202])

In [ ]:
# save the true counts here
image_names_blk_0202 = [file[0] for file in function_results_0202]

In [ ]:
true_counts_df_block_0202 = pd.DataFrame({'Image_name' :image_names_blk_0202, 'True_count': true_counts_block_0202, 'Convolved_count': convolved_counts_block_0202})

In [ ]:
true_counts_df_block_0202

In [ ]:
# save this dataframe
true_counts_df_block_0202.to_csv("All_data_alt/test_true_counts/true_counts_blk_0202.csv", index = False)

In [ ]:
im_height = all_im_sizes_0202[0][0]
im_width = all_im_sizes_0202[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0202, response_name_list_0202 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0202)

In [ ]:
stacked_responses_0202.shape

In [ ]:
response_name_list_0202[0], response_name_list_0202[1], response_name_list_0202[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0202/density_maps_seqs_0202.npy', stacked_responses_0202)

Block 0203

In [ ]:
# location of the files
block_0203 = '../../../Spring_2024/S_lab_TasselNet/Block_9_TN/Block_9_images_and_xml'

In [ ]:
task_spec_im_files_0203, task_spec_xml_files_0203, mean_0203 = chose_xml_and_jpeg(block_0203)

In [ ]:
# task_spec_im_files_0203

In [ ]:
# task_spec_xml_files_0203

In [ ]:
# mean_0203

In [ ]:
images_0203 = []
for file in task_spec_im_files_0203:
    joined_path = os.path.join(block_0203, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0203.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0203[0].shape[0]
image_weight = images_0203[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0203, sub_window_name_list_0203 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0203)

In [ ]:
len(sub_window_name_list_0203)

In [ ]:
sub_window_name_list_0203[0], sub_window_name_list_0203[1], sub_window_name_list_0203[-1]

In [ ]:
stacked_sub_windows_0203.shape

In [ ]:
np.save('All_data_alt/block_0203/subwindow_seqs_0203.npy', stacked_sub_windows_0203)

In [ ]:
# xml files
task_spec_xml_files_0203

In [ ]:
# Corresponding image files
response_image_files_0203 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0203]

In [ ]:
response_image_files_0203

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0203.sort()
response_image_files_0203.sort()

In [ ]:
len(task_spec_xml_files_0203), len(response_image_files_0203)

In [ ]:
old_path = block_0203
old_path

In [ ]:
new_store_path_0203 = 'All_data_alt/block_0203/all_np_files'

In [ ]:
%%time
all_im_sizes_0203 = []
for file in response_image_files_0203:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0203)
    all_im_sizes_0203.append(im_size)

In [ ]:
all_im_sizes_0203

In [ ]:
# we need file names
block_0203_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0203]
block_0203_response_file_names

In [ ]:
new_store_path_0203

In [ ]:
block_0203

In [ ]:
%%time
# let's run the function
function_results_0203 = []
for file_name in block_0203_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0203, block_0203, new_store_path_0203)
    function_results_0203.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0203 = [file[2] for file in function_results_0203]
convolved_counts_block_0203 = [file[3] for file in function_results_0203]

In [ ]:
true_counts_block_0203

In [ ]:
convolved_counts_block_0203

In [ ]:
np.mean(true_counts_block_0203 == np.round(convolved_counts_block_0203))

In [ ]:
all_density_maps_0203 = [file[-1] for file in function_results_0203]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0203 == [np.sum(file) for file in all_density_maps_0203])

In [ ]:
# save the true counts here
image_names_blk_0203 = [file[0] for file in function_results_0203]

In [ ]:
true_counts_df_block_0203 = pd.DataFrame({'Image_name' :image_names_blk_0203, 'True_count': true_counts_block_0203, 'Convolved_count': convolved_counts_block_0203})

In [ ]:
true_counts_df_block_0203

In [ ]:
# save this dataframe
true_counts_df_block_0203.to_csv("All_data_alt/test_true_counts/true_counts_blk_0203.csv", index = False)

In [ ]:
im_height = all_im_sizes_0203[0][0]
im_width = all_im_sizes_0203[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0203, response_name_list_0203 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0203)

In [ ]:
stacked_responses_0203.shape

In [ ]:
response_name_list_0203[0], response_name_list_0203[1], response_name_list_0203[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0203/density_maps_seqs_0203.npy', stacked_responses_0203)

Block 0204

In [ ]:
# location of the files
block_0204 = '../../../Spring_2024/S_lab_TasselNet/Block_10_TN/Block_10_images_and_xml'

In [ ]:
task_spec_im_files_0204, task_spec_xml_files_0204, mean_0204 = chose_xml_and_jpeg(block_0204)

In [ ]:
# task_spec_im_files_0204

In [ ]:
# task_spec_xml_files_0204

In [ ]:
# mean_0204

In [ ]:
images_0204 = []
for file in task_spec_im_files_0204:
    joined_path = os.path.join(block_0204, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0204.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0204[0].shape[0]
image_weight = images_0204[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0204, sub_window_name_list_0204 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0204)

In [ ]:
len(sub_window_name_list_0204)

In [ ]:
sub_window_name_list_0204[0], sub_window_name_list_0204[1], sub_window_name_list_0204[-1]

In [ ]:
stacked_sub_windows_0204.shape

In [ ]:
np.save('All_data_alt/block_0204/subwindow_seqs_0204.npy', stacked_sub_windows_0204)

In [ ]:
# xml files
task_spec_xml_files_0204

In [ ]:
# Corresponding image files
response_image_files_0204 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0204]

In [ ]:
response_image_files_0204

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0204.sort()
response_image_files_0204.sort()

In [ ]:
len(task_spec_xml_files_0204), len(response_image_files_0204)

In [ ]:
old_path = block_0204
old_path

In [ ]:
new_store_path_0204 = 'All_data_alt/block_0204/all_np_files'

In [ ]:
%%time
all_im_sizes_0204 = []
for file in response_image_files_0204:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0204)
    all_im_sizes_0204.append(im_size)

In [ ]:
all_im_sizes_0204

In [ ]:
# we need file names
block_0204_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0204]
block_0204_response_file_names

In [ ]:
new_store_path_0204

In [ ]:
block_0204

In [ ]:
%%time
# let's run the function
function_results_0204 = []
for file_name in block_0204_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0204, block_0204, new_store_path_0204)
    function_results_0204.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0204 = [file[2] for file in function_results_0204]
convolved_counts_block_0204 = [file[3] for file in function_results_0204]

In [ ]:
true_counts_block_0204

In [ ]:
convolved_counts_block_0204

In [ ]:
np.mean(true_counts_block_0204 == np.round(convolved_counts_block_0204))

In [ ]:
all_density_maps_0204 = [file[-1] for file in function_results_0204]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0204 == [np.sum(file) for file in all_density_maps_0204])

In [ ]:
# save the true counts here
image_names_blk_0204 = [file[0] for file in function_results_0204]

In [ ]:
true_counts_df_block_0204 = pd.DataFrame({'Image_name' :image_names_blk_0204, 'True_count': true_counts_block_0204, 'Convolved_count': convolved_counts_block_0204})

In [ ]:
true_counts_df_block_0204

In [ ]:
# save this dataframe
true_counts_df_block_0204.to_csv("All_data_alt/test_true_counts/true_counts_blk_0204.csv", index = False)

In [ ]:
im_height = all_im_sizes_0204[0][0]
im_width = all_im_sizes_0204[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0204, response_name_list_0204 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0204)

In [ ]:
stacked_responses_0204.shape

In [ ]:
response_name_list_0204[0], response_name_list_0204[1], response_name_list_0204[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0204/density_maps_seqs_0204.npy', stacked_responses_0204)

Block 0205

In [ ]:
# location of the files
block_0205 = '../../../Spring_2024/S_lab_TasselNet/Block_11_TN/Block_11_images_and_xml'

In [ ]:
task_spec_im_files_0205, task_spec_xml_files_0205, mean_0205 = chose_xml_and_jpeg(block_0205)

In [ ]:
# task_spec_im_files_0205

In [ ]:
# task_spec_xml_files_0205

In [ ]:
# mean_0205

In [ ]:
images_0205 = []
for file in task_spec_im_files_0205:
    joined_path = os.path.join(block_0205, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0205.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0205[0].shape[0]
image_weight = images_0205[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0205, sub_window_name_list_0205 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0205)

In [ ]:
len(sub_window_name_list_0205)

In [ ]:
sub_window_name_list_0205[0], sub_window_name_list_0205[1], sub_window_name_list_0205[-1]

In [ ]:
stacked_sub_windows_0205.shape

In [ ]:
np.save('All_data_alt/block_0205/subwindow_seqs_0205.npy', stacked_sub_windows_0205)

In [ ]:
# xml files
task_spec_xml_files_0205

In [ ]:
# Corresponding image files
response_image_files_0205 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0205]

In [ ]:
response_image_files_0205

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0205.sort()
response_image_files_0205.sort()

In [ ]:
len(task_spec_xml_files_0205), len(response_image_files_0205)

In [ ]:
old_path = block_0205
old_path

In [ ]:
new_store_path_0205 = 'All_data_alt/block_0205/all_np_files'

In [ ]:
%%time
all_im_sizes_0205 = []
for file in response_image_files_0205:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0205)
    all_im_sizes_0205.append(im_size)

In [ ]:
all_im_sizes_0205

In [ ]:
# we need file names
block_0205_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0205]
block_0205_response_file_names

In [ ]:
new_store_path_0205

In [ ]:
block_0205

In [ ]:
%%time
# let's run the function
function_results_0205 = []
for file_name in block_0205_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0205, block_0205, new_store_path_0205)
    function_results_0205.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0205 = [file[2] for file in function_results_0205]
convolved_counts_block_0205 = [file[3] for file in function_results_0205]

In [ ]:
true_counts_block_0205

In [ ]:
convolved_counts_block_0205

In [ ]:
np.mean(true_counts_block_0205 == np.round(convolved_counts_block_0205))

In [ ]:
all_density_maps_0205 = [file[-1] for file in function_results_0205]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0205 == [np.sum(file) for file in all_density_maps_0205])

In [ ]:
# save the true counts here
image_names_blk_0205 = [file[0] for file in function_results_0205]

In [ ]:
true_counts_df_block_0205 = pd.DataFrame({'Image_name' :image_names_blk_0205, 'True_count': true_counts_block_0205, 'Convolved_count': convolved_counts_block_0205})

In [ ]:
true_counts_df_block_0205

In [ ]:
# save this dataframe
true_counts_df_block_0205.to_csv("All_data_alt/test_true_counts/true_counts_blk_0205.csv", index = False)

In [ ]:
im_height = all_im_sizes_0205[0][0]
im_width = all_im_sizes_0205[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0205, response_name_list_0205 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0205)

In [ ]:
stacked_responses_0205.shape

In [ ]:
response_name_list_0205[0], response_name_list_0205[1], response_name_list_0205[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0205/density_maps_seqs_0205.npy', stacked_responses_0205)

Block 0206

In [ ]:
# location of the files
block_0206 = '../../../Spring_2024/S_lab_TasselNet/Block_12_TN/Block_12_images_and_xml'

In [ ]:
task_spec_im_files_0206, task_spec_xml_files_0206, mean_0206 = chose_xml_and_jpeg(block_0206)

In [ ]:
# task_spec_im_files_0206

In [ ]:
# task_spec_xml_files_0206

In [ ]:
# mean_0206

In [ ]:
images_0206 = []
for file in task_spec_im_files_0206:
    joined_path = os.path.join(block_0206, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0206.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0206[0].shape[0]
image_weight = images_0206[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0206, sub_window_name_list_0206 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0206)

In [ ]:
len(sub_window_name_list_0206)

In [ ]:
sub_window_name_list_0206[0], sub_window_name_list_0206[1], sub_window_name_list_0206[-1]

In [ ]:
stacked_sub_windows_0206.shape

In [ ]:
np.save('All_data_alt/block_0206/subwindow_seqs_0206.npy', stacked_sub_windows_0206)

In [ ]:
# xml files
task_spec_xml_files_0206

In [ ]:
# Corresponding image files
response_image_files_0206 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0206]

In [ ]:
response_image_files_0206

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0206.sort()
response_image_files_0206.sort()

In [ ]:
len(task_spec_xml_files_0206), len(response_image_files_0206)

In [ ]:
old_path = block_0206
old_path

In [ ]:
new_store_path_0206 = 'All_data_alt/block_0206/all_np_files'

In [ ]:
%%time
all_im_sizes_0206 = []
for file in response_image_files_0206:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0206)
    all_im_sizes_0206.append(im_size)

In [ ]:
all_im_sizes_0206

In [ ]:
# we need file names
block_0206_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0206]
block_0206_response_file_names

In [ ]:
new_store_path_0206

In [ ]:
block_0206

In [ ]:
%%time
# let's run the function
function_results_0206 = []
for file_name in block_0206_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0206, block_0206, new_store_path_0206)
    function_results_0206.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0206 = [file[2] for file in function_results_0206]
convolved_counts_block_0206 = [file[3] for file in function_results_0206]

In [ ]:
true_counts_block_0206

In [ ]:
convolved_counts_block_0206

In [ ]:
np.mean(true_counts_block_0206 == np.round(convolved_counts_block_0206))

In [ ]:
all_density_maps_0206 = [file[-1] for file in function_results_0206]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0206 == [np.sum(file) for file in all_density_maps_0206])

In [ ]:
# save the true counts here
image_names_blk_0206 = [file[0] for file in function_results_0206]

In [ ]:
true_counts_df_block_0206 = pd.DataFrame({'Image_name' :image_names_blk_0206, 'True_count': true_counts_block_0206, 'Convolved_count': convolved_counts_block_0206})

In [ ]:
true_counts_df_block_0206

In [ ]:
# save this dataframe
true_counts_df_block_0206.to_csv("All_data_alt/test_true_counts/true_counts_blk_0206.csv", index = False)

In [ ]:
im_height = all_im_sizes_0206[0][0]
im_width = all_im_sizes_0206[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0206, response_name_list_0206 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0206)

In [ ]:
stacked_responses_0206.shape

In [ ]:
response_name_list_0206[0], response_name_list_0206[1], response_name_list_0206[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0206/density_maps_seqs_0206.npy', stacked_responses_0206)

Block 0301

In [ ]:
# location of the files
block_0301 = '../../../Spring_2024/S_lab_TasselNet/Block_13_TN/Block_13_images_and_xml'

In [ ]:
task_spec_im_files_0301, task_spec_xml_files_0301, mean_0301 = chose_xml_and_jpeg(block_0301)

In [ ]:
# task_spec_im_files_0301

In [ ]:
# task_spec_xml_files_0301

In [ ]:
# mean_0301

In [ ]:
images_0301 = []
for file in task_spec_im_files_0301:
    joined_path = os.path.join(block_0301, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0301.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0301[0].shape[0]
image_weight = images_0301[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0301, sub_window_name_list = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0301)

In [ ]:
len(sub_window_name_list)

In [ ]:
sub_window_name_list[0], sub_window_name_list[1], sub_window_name_list[-1]

In [ ]:
stacked_sub_windows_0301.shape

In [ ]:
np.save('All_data_alt/block_0301/subwindow_seqs_0301.npy', stacked_sub_windows_0301)

In [ ]:
# xml files
task_spec_xml_files_0301

In [ ]:
# Corresponding image files
response_image_files_0301 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0301]

In [ ]:
response_image_files_0301

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0301.sort()
response_image_files_0301.sort()

In [ ]:
len(task_spec_xml_files_0301), len(response_image_files_0301)

In [ ]:
old_path = block_0301
old_path

In [ ]:
new_store_path_0301 = 'All_data_alt/block_0301/all_np_files'

In [ ]:
%%time
all_im_sizes_0301 = []
for file in response_image_files_0301:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0301)
    all_im_sizes_0301.append(im_size)

In [ ]:
all_im_sizes_0301

In [ ]:
# we need file names
block_0301_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0301]
block_0301_response_file_names

In [ ]:
new_store_path_0301

In [ ]:
block_0301

In [ ]:
%%time
# let's run the function
function_results_0301 = []
for file_name in block_0301_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0301, block_0301, new_store_path_0301)
    function_results_0301.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0301 = [file[2] for file in function_results_0301]
convolved_counts_block_0301 = [file[3] for file in function_results_0301]

In [ ]:
true_counts_block_0301

In [ ]:
convolved_counts_block_0301

In [ ]:
np.mean(true_counts_block_0301 == np.round(convolved_counts_block_0301))

In [ ]:
all_density_maps_0301 = [file[-1] for file in function_results_0301]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0301 == [np.sum(file) for file in all_density_maps_0301])

In [ ]:
# save the true counts here
image_names_blk_0301 = [file[0] for file in function_results_0301]

In [ ]:
true_counts_df_block_0301 = pd.DataFrame({'Image_name' :image_names_blk_0301, 'True_count': true_counts_block_0301, 'Convolved_count': convolved_counts_block_0301})

In [ ]:
true_counts_df_block_0301

In [ ]:
# save this dataframe
true_counts_df_block_0301.to_csv("All_data_alt/test_true_counts/true_counts_blk_0301.csv", index = False)

In [ ]:
im_height = all_im_sizes_0301[0][0]
im_width = all_im_sizes_0301[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0301, response_name_list_0301 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0301)

In [ ]:
stacked_responses_0301.shape

In [ ]:
response_name_list_0301[0], response_name_list_0301[1], response_name_list_0301[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0301/density_maps_seqs_0301.npy', stacked_responses_0301)

Block 0302

In [ ]:
# location of the files
block_0302 = '../../../Spring_2024/S_lab_TasselNet/Block_14_TN/Block_14_images_and_xml'

In [ ]:
task_spec_im_files_0302, task_spec_xml_files_0302, mean_0302 = chose_xml_and_jpeg(block_0302)

In [ ]:
# task_spec_im_files_0302

In [ ]:
# task_spec_xml_files_0302

In [ ]:
# mean_0302

In [ ]:
images_0302 = []
for file in task_spec_im_files_0302:
    joined_path = os.path.join(block_0302, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0302.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0302[0].shape[0]
image_weight = images_0302[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0302, sub_window_name_list = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0302)

In [ ]:
len(sub_window_name_list)

In [ ]:
sub_window_name_list[0], sub_window_name_list[1], sub_window_name_list[-1]

In [ ]:
stacked_sub_windows_0302.shape

In [ ]:
np.save('All_data_alt/block_0302/subwindow_seqs_0302.npy', stacked_sub_windows_0302)

In [ ]:
# xml files
task_spec_xml_files_0302

In [ ]:
# Corresponding image files
response_image_files_0302 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0302]

In [ ]:
response_image_files_0302

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0302.sort()
response_image_files_0302.sort()

In [ ]:
len(task_spec_xml_files_0302), len(response_image_files_0302)

In [ ]:
old_path = block_0302
old_path

In [ ]:
new_store_path_0302 = 'All_data_alt/block_0302/all_np_files'

In [ ]:
%%time
all_im_sizes_0302 = []
for file in response_image_files_0302:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0302)
    all_im_sizes_0302.append(im_size)

In [ ]:
all_im_sizes_0302

In [ ]:
# we need file names
block_0302_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0302]
block_0302_response_file_names

In [ ]:
new_store_path_0302

In [ ]:
block_0302

In [ ]:
%%time
# let's run the function
function_results_0302 = []
for file_name in block_0302_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0302, block_0302, new_store_path_0302)
    function_results_0302.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0302 = [file[2] for file in function_results_0302]
convolved_counts_block_0302 = [file[3] for file in function_results_0302]

In [ ]:
true_counts_block_0302

In [ ]:
convolved_counts_block_0302

In [ ]:
np.mean(true_counts_block_0302 == np.round(convolved_counts_block_0302))

In [ ]:
all_density_maps_0302 = [file[-1] for file in function_results_0302]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0302 == [np.sum(file) for file in all_density_maps_0302])

In [ ]:
# save the true counts here
image_names_blk_0302 = [file[0] for file in function_results_0302]

In [ ]:
true_counts_df_block_0302 = pd.DataFrame({'Image_name' :image_names_blk_0302, 'True_count': true_counts_block_0302, 'Convolved_count': convolved_counts_block_0302})

In [ ]:
true_counts_df_block_0302

In [ ]:
# save this dataframe
true_counts_df_block_0302.to_csv("All_data_alt/test_true_counts/true_counts_blk_0302.csv", index = False)

In [ ]:
im_height = all_im_sizes_0302[0][0]
im_width = all_im_sizes_0302[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0302, response_name_list_0302 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0302)

In [ ]:
stacked_responses_0302.shape

In [ ]:
response_name_list_0302[0], response_name_list_0302[1], response_name_list_0302[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0302/density_maps_seqs_0302.npy', stacked_responses_0302)

Block 0303

In [ ]:
# location of the files
block_0303 = '../../../Spring_2024/S_lab_TasselNet/Block_15_TN/Block_15_images_and_xml'

In [ ]:
task_spec_im_files_0303, task_spec_xml_files_0303, mean_0303 = chose_xml_and_jpeg(block_0303)

In [ ]:
# task_spec_im_files_0303

In [ ]:
# task_spec_xml_files_0303

In [ ]:
# mean_0303

In [ ]:
images_0303 = []
for file in task_spec_im_files_0303:
    joined_path = os.path.join(block_0303, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0303.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0303[0].shape[0]
image_weight = images_0303[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0303, sub_window_name_list_0303 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0303)

In [ ]:
len(sub_window_name_list_0303)

In [ ]:
sub_window_name_list_0303[0], sub_window_name_list_0303[1], sub_window_name_list_0303[-1]

In [ ]:
stacked_sub_windows_0303.shape

In [ ]:
np.save('All_data_alt/block_0303/subwindow_seqs_0303.npy', stacked_sub_windows_0303)

In [ ]:
# xml files
task_spec_xml_files_0303

In [ ]:
# Corresponding image files
response_image_files_0303 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0303]

In [ ]:
response_image_files_0303

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0303.sort()
response_image_files_0303.sort()

In [ ]:
len(task_spec_xml_files_0303), len(response_image_files_0303)

In [ ]:
old_path = block_0303
old_path

In [ ]:
new_store_path_0303 = 'All_data_alt/block_0303/all_np_files'

In [ ]:
%%time
all_im_sizes_0303 = []
for file in response_image_files_0303:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0303)
    all_im_sizes_0303.append(im_size)

In [ ]:
all_im_sizes_0303

In [ ]:
# we need file names
block_0303_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0303]
block_0303_response_file_names

In [ ]:
new_store_path_0303

In [ ]:
block_0303

In [ ]:
%%time
# let's run the function
function_results_0303 = []
for file_name in block_0303_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0303, block_0303, new_store_path_0303)
    function_results_0303.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0303 = [file[2] for file in function_results_0303]
convolved_counts_block_0303 = [file[3] for file in function_results_0303]

In [ ]:
true_counts_block_0303

In [ ]:
convolved_counts_block_0303

In [ ]:
np.mean(true_counts_block_0303 == np.round(convolved_counts_block_0303))

In [ ]:
all_density_maps_0303 = [file[-1] for file in function_results_0303]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0303 == [np.sum(file) for file in all_density_maps_0303])

In [ ]:
# save the true counts here
image_names_blk_0303 = [file[0] for file in function_results_0303]

In [ ]:
true_counts_df_block_0303 = pd.DataFrame({'Image_name' :image_names_blk_0303, 'True_count': true_counts_block_0303, 'Convolved_count': convolved_counts_block_0303})

In [ ]:
true_counts_df_block_0303

In [ ]:
# save this dataframe
true_counts_df_block_0303.to_csv("All_data_alt/test_true_counts/true_counts_blk_0303.csv", index = False)

In [ ]:
im_height = all_im_sizes_0303[0][0]
im_width = all_im_sizes_0303[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0303, response_name_list_0303 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0303)

In [ ]:
stacked_responses_0303.shape

In [ ]:
response_name_list_0303[0], response_name_list_0303[1], response_name_list_0303[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0303/density_maps_seqs_0303.npy', stacked_responses_0303)

Block 0304

In [ ]:
# location of the files
block_0304 = '../../../Spring_2024/S_lab_TasselNet/Block_16_TN/Block_16_images_and_xml'

In [ ]:
task_spec_im_files_0304, task_spec_xml_files_0304, mean_0304 = chose_xml_and_jpeg(block_0304)

In [ ]:
# task_spec_im_files_0304

In [ ]:
# task_spec_xml_files_0304

In [ ]:
# mean_0304

In [ ]:
images_0304 = []
for file in task_spec_im_files_0304:
    joined_path = os.path.join(block_0304, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0304.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0304[0].shape[0]
image_weight = images_0304[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0304, sub_window_name_list_0304 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0304)

In [ ]:
len(sub_window_name_list_0304)

In [ ]:
sub_window_name_list_0304[0], sub_window_name_list_0304[1], sub_window_name_list_0304[-1]

In [ ]:
stacked_sub_windows_0304.shape

In [ ]:
np.save('All_data_alt/block_0304/subwindow_seqs_0304.npy', stacked_sub_windows_0304)

In [ ]:
# xml files
task_spec_xml_files_0304

In [ ]:
# Corresponding image files
response_image_files_0304 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0304]

In [ ]:
response_image_files_0304

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0304.sort()
response_image_files_0304.sort()

In [ ]:
len(task_spec_xml_files_0304), len(response_image_files_0304)

In [ ]:
old_path = block_0304
old_path

In [ ]:
new_store_path_0304 = 'All_data_alt/block_0304/all_np_files'

In [ ]:
%%time
all_im_sizes_0304 = []
for file in response_image_files_0304:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0304)
    all_im_sizes_0304.append(im_size)

In [ ]:
all_im_sizes_0304

In [ ]:
# we need file names
block_0304_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0304]
block_0304_response_file_names

In [ ]:
new_store_path_0304

In [ ]:
block_0304

In [ ]:
%%time
# let's run the function
function_results_0304 = []
for file_name in block_0304_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0304, block_0304, new_store_path_0304)
    function_results_0304.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0304 = [file[2] for file in function_results_0304]
convolved_counts_block_0304 = [file[3] for file in function_results_0304]

In [ ]:
true_counts_block_0304

In [ ]:
convolved_counts_block_0304

In [ ]:
np.mean(true_counts_block_0304 == np.round(convolved_counts_block_0304))

In [ ]:
all_density_maps_0304 = [file[-1] for file in function_results_0304]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0304 == [np.sum(file) for file in all_density_maps_0304])

In [ ]:
# save the true counts here
image_names_blk_0304 = [file[0] for file in function_results_0304]

In [ ]:
true_counts_df_block_0304 = pd.DataFrame({'Image_name' :image_names_blk_0304, 'True_count': true_counts_block_0304, 'Convolved_count': convolved_counts_block_0304})

In [ ]:
true_counts_df_block_0304

In [ ]:
# save this dataframe
true_counts_df_block_0304.to_csv("All_data_alt/test_true_counts/true_counts_blk_0304.csv", index = False)

In [ ]:
im_height = all_im_sizes_0304[0][0]
im_width = all_im_sizes_0304[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0304, response_name_list_0304 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0304)

In [ ]:
stacked_responses_0304.shape

In [ ]:
response_name_list_0304[0], response_name_list_0304[1], response_name_list_0304[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0304/density_maps_seqs_0304.npy', stacked_responses_0304)

Block 0305

In [ ]:
# location of the files
block_0305 = '../../../Spring_2024/S_lab_TasselNet/Block_17_TN/Block_17_images_and_xml'

In [ ]:
task_spec_im_files_0305, task_spec_xml_files_0305, mean_0305 = chose_xml_and_jpeg(block_0305)

In [ ]:
# task_spec_im_files_0305

In [ ]:
# task_spec_xml_files_0305

In [ ]:
# mean_0305

In [ ]:
images_0305 = []
for file in task_spec_im_files_0305:
    joined_path = os.path.join(block_0305, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0305.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0305[0].shape[0]
image_weight = images_0305[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0305, sub_window_name_list_0305 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0305)

In [ ]:
len(sub_window_name_list_0305)

In [ ]:
sub_window_name_list_0305[0], sub_window_name_list_0305[1], sub_window_name_list_0305[-1]

In [ ]:
stacked_sub_windows_0305.shape

In [ ]:
np.save('All_data_alt/block_0305/subwindow_seqs_0305.npy', stacked_sub_windows_0305)

In [ ]:
# xml files
task_spec_xml_files_0305

In [ ]:
# Corresponding image files
response_image_files_0305 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0305]

In [ ]:
response_image_files_0305

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0305.sort()
response_image_files_0305.sort()

In [ ]:
len(task_spec_xml_files_0305), len(response_image_files_0305)

In [ ]:
old_path = block_0305
old_path

In [ ]:
new_store_path_0305 = 'All_data_alt/block_0305/all_np_files'

In [ ]:
%%time
all_im_sizes_0305 = []
for file in response_image_files_0305:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0305)
    all_im_sizes_0305.append(im_size)

In [ ]:
all_im_sizes_0305

In [ ]:
# we need file names
block_0305_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0305]
block_0305_response_file_names

In [ ]:
new_store_path_0305

In [ ]:
block_0305

In [ ]:
%%time
# let's run the function
function_results_0305 = []
for file_name in block_0305_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0305, block_0305, new_store_path_0305)
    function_results_0305.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0305 = [file[2] for file in function_results_0305]
convolved_counts_block_0305 = [file[3] for file in function_results_0305]

In [ ]:
true_counts_block_0305

In [ ]:
convolved_counts_block_0305

In [ ]:
np.mean(true_counts_block_0305 == np.round(convolved_counts_block_0305))

In [ ]:
all_density_maps_0305 = [file[-1] for file in function_results_0305]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0305 == [np.sum(file) for file in all_density_maps_0305])

In [ ]:
# save the true counts here
image_names_blk_0305 = [file[0] for file in function_results_0305]

In [ ]:
true_counts_df_block_0305 = pd.DataFrame({'Image_name' :image_names_blk_0305, 'True_count': true_counts_block_0305, 'Convolved_count': convolved_counts_block_0305})

In [ ]:
true_counts_df_block_0305

In [ ]:
# save this dataframe
true_counts_df_block_0305.to_csv("All_data_alt/test_true_counts/true_counts_blk_0305.csv", index = False)

In [ ]:
im_height = all_im_sizes_0305[0][0]
im_width = all_im_sizes_0305[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0305, response_name_list_0305 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0305)

In [ ]:
stacked_responses_0305.shape

In [ ]:
response_name_list_0305[0], response_name_list_0305[1], response_name_list_0305[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0305/density_maps_seqs_0305.npy', stacked_responses_0305)

Block 0306

In [ ]:
# location of the files
block_0306 = '../../../Spring_2024/S_lab_TasselNet/Block_18_TN/Block_18_images_and_xml'

In [ ]:
task_spec_im_files_0306, task_spec_xml_files_0306, mean_0306 = chose_xml_and_jpeg(block_0306)

In [ ]:
# task_spec_im_files_0306

In [ ]:
# task_spec_xml_files_0306

In [ ]:
# mean_0306

In [ ]:
images_0306 = []
for file in task_spec_im_files_0306:
    joined_path = os.path.join(block_0306, file)
    read_image = plt.imread(joined_path)
    plt.imshow(read_image)
    plt.show()
    images_0306.append(read_image)

In [ ]:
%%time

# get the function output

image_height = images_0306[0].shape[0]
image_weight = images_0306[0].shape[1]
stride = 8
kernel_size = 32
n_channels =3

stacked_sub_windows_0306, sub_window_name_list_0306 = create_and_stack_subwindows(image_height, image_weight, stride, kernel_size, n_channels, images_0306)

In [ ]:
len(sub_window_name_list_0306)

In [ ]:
sub_window_name_list_0306[0], sub_window_name_list_0306[1], sub_window_name_list_0306[-1]

In [ ]:
stacked_sub_windows_0306.shape

In [ ]:
np.save('All_data_alt/block_0306/subwindow_seqs_0306.npy', stacked_sub_windows_0306)

In [ ]:
# xml files
task_spec_xml_files_0306

In [ ]:
# Corresponding image files
response_image_files_0306 = [file.split('.')[0] + '.jpeg' for file in task_spec_xml_files_0306]

In [ ]:
response_image_files_0306

In [ ]:
# sort the file lists first just in case
task_spec_xml_files_0306.sort()
response_image_files_0306.sort()

In [ ]:
len(task_spec_xml_files_0306), len(response_image_files_0306)

In [ ]:
old_path = block_0306
old_path

In [ ]:
new_store_path_0306 = 'All_data_alt/block_0306/all_np_files'

In [ ]:
%%time
all_im_sizes_0306 = []
for file in response_image_files_0306:
    im_size = store_images_as_np_arrays_horizontal(old_path, file, new_store_path_0306)
    all_im_sizes_0306.append(im_size)

In [ ]:
all_im_sizes_0306

In [ ]:
# we need file names
block_0306_response_file_names = [file.split('.')[0] for file in task_spec_xml_files_0306]
block_0306_response_file_names

In [ ]:
new_store_path_0306

In [ ]:
block_0306

In [ ]:
%%time
# let's run the function
function_results_0306 = []
for file_name in block_0306_response_file_names:
    result = get_density_maps_horizontal(file_name, new_store_path_0306, block_0306, new_store_path_0306)
    function_results_0306.append(result)

In [ ]:
# let's first map the true and the convolved counts
true_counts_block_0306 = [file[2] for file in function_results_0306]
convolved_counts_block_0306 = [file[3] for file in function_results_0306]

In [ ]:
true_counts_block_0306

In [ ]:
convolved_counts_block_0306

In [ ]:
np.mean(true_counts_block_0306 == np.round(convolved_counts_block_0306))

In [ ]:
all_density_maps_0306 = [file[-1] for file in function_results_0306]

In [ ]:
# one more sanity check - to make sure the convolved sums match with the density map sums
np.mean(convolved_counts_block_0306 == [np.sum(file) for file in all_density_maps_0306])

In [ ]:
# save the true counts here
image_names_blk_0306 = [file[0] for file in function_results_0306]

In [ ]:
true_counts_df_block_0306 = pd.DataFrame({'Image_name' :image_names_blk_0306, 'True_count': true_counts_block_0306, 'Convolved_count': convolved_counts_block_0306})

In [ ]:
true_counts_df_block_0306

In [ ]:
# save this dataframe
true_counts_df_block_0306.to_csv("All_data_alt/test_true_counts/true_counts_blk_0306.csv", index = False)

In [ ]:
im_height = all_im_sizes_0306[0][0]
im_width = all_im_sizes_0306[0][1]
stride = 8
kernel_size = 32
im_height, im_width

In [ ]:
%%time
# see if this works
stacked_responses_0306, response_name_list_0306 = tassel_density_sequences(im_height, im_width, stride, kernel_size, all_density_maps_0306)

In [ ]:
stacked_responses_0306.shape

In [ ]:
response_name_list_0306[0], response_name_list_0306[1], response_name_list_0306[-1]

In [ ]:
# save the stacked response

In [ ]:
np.save('All_data_alt/block_0306/density_maps_seqs_0306.npy', stacked_responses_0306)